In [96]:
import pandas as pd

In [97]:
df_train=pd.read_csv('./cdata/c_train.csv')
df_test=pd.read_csv('./cdata/c_test.csv')

In [98]:
df_train.columns

Index(['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Transported', 'HomePlanet_Europa', 'HomePlanet_Mars',
       'HomePlanet_Unknown', 'CryoSleep_True', 'CryoSleep_Unknown',
       'Destination_PSO J318.5-22', 'Destination_TRAPPIST-1e',
       'Destination_Unknown', 'VIP_True', 'VIP_Unknown'],
      dtype='str')

In [99]:
df_test.shape

(4277, 17)

In [100]:
df_train.isna().sum().sum()==df_test.isna().sum().sum()==0

np.True_

In [101]:
df_test.shape[1]==df_train.shape[1]-1

False

In [102]:
X=df_train.drop(columns=['Transported'])
y=df_train['Transported']

In [103]:
'Transported' in df_test.columns

False

In [104]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

In [60]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
from sklearn.metrics import accuracy_score

In [64]:
lb= lgb.LGBMClassifier(random_state=42)
lb.fit(X_train, y_train)
print(f"LightGBM Accuracy: {accuracy_score(y_val, lb.predict(X_val))}")

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 2924, number of negative: 2888
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000297 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1372
[LightGBM] [Info] Number of data points in the train set: 5812, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503097 -> initscore=0.012388
[LightGBM] [Info] Start training from score 0.012388
LightGBM Accuracy: 0.7770130763936682


In [49]:
dt=DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
print(f"Decision Tree Accuracy: {accuracy_score(y_val, dt.predict(X_val))}")

Decision Tree Accuracy: 0.7336545079146594


In [50]:
rf=RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
print(f"Random Forest Accuracy: {accuracy_score(y_val, rf.predict(X_val))}")

Random Forest Accuracy: 0.7728836889194769


In [51]:
gb=GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
print(f"Gradient Boosting Accuracy: {accuracy_score(y_val, gb.predict(X_val))}")

Gradient Boosting Accuracy: 0.7804542326221611


In [52]:
xgb=XGBClassifier(random_state=42)
xgb.fit(X_train, y_train)
print(f"XGBoost Accuracy: {accuracy_score(y_val, xgb.predict(X_val))}")

XGBoost Accuracy: 0.7680660701995871


In [55]:
voting_clf = VotingClassifier(estimators=[('dt', dt), ('rf', rf), ('gb', gb), ('xgb', xgb)], voting='hard')
voting_clf.fit(X_train, y_train)
print(f"Voting Classifier Accuracy at hard voting: {accuracy_score(y_val, voting_clf.predict(X_val))}")

Voting Classifier Accuracy at hard voting: 0.7728836889194769


In [56]:
voting_clf = VotingClassifier(estimators=[('dt', dt), ('rf', rf), ('gb', gb), ('xgb', xgb)], voting='soft')
voting_clf.fit(X_train, y_train)
print(f"Voting Classifier Accuracy at soft voting: {accuracy_score(y_val, voting_clf.predict(X_val))}")

Voting Classifier Accuracy at soft voting: 0.7749483826565726


In [58]:
voting_clf = VotingClassifier(estimators=[('dt', dt), ('rf', rf), ('gb', gb), ('xgb', xgb)], voting='hard', weights=[1, 2, 3, 4])
voting_clf.fit(X_train, y_train)
print("Weighted Voting Accuracy at hard voting:", accuracy_score(y_val, voting_clf.predict(X_val)))

Weighted Voting Accuracy at hard voting: 0.7777013076393668


In [59]:
voting_clf = VotingClassifier(estimators=[('dt', dt), ('rf', rf), ('gb', gb), ('xgb', xgb)], voting='soft', weights=[1, 2, 3, 4])
voting_clf.fit(X_train, y_train)
print("Weighted Voting Accuracy at soft voting:", accuracy_score(y_val, voting_clf.predict(X_val)))

Weighted Voting Accuracy at soft voting: 0.7770130763936682


In [65]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0],
    'max_features': ['sqrt', 'log2', None]
}
grid_search = GridSearchCV(
    estimator=gb,
    param_grid=param_grid,
    cv=5,            
    scoring='accuracy',
    n_jobs=-1       
)

In [66]:
grid_search.fit(X_train, y_train)
print("Best Parameters:", grid_search.best_params_)
print("Best CV Accuracy:", grid_search.best_score_)
print("Validation Accuracy:", accuracy_score(y_val, grid_search.best_estimator_.predict(X_val)))

Best Parameters: {'learning_rate': 0.05, 'max_depth': 5, 'max_features': None, 'n_estimators': 500, 'subsample': 0.8}
Best CV Accuracy: 0.8023072266957524
Validation Accuracy: 0.7715072264280798


In [67]:
from sklearn.model_selection import RandomizedSearchCV


param_dist = {
    'num_leaves': [31, 63, 127],
    'max_depth': [-1, 5, 10],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'n_estimators': [100, 300, 600],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_samples': [10, 20, 50],
    'reg_alpha': [0.0, 0.1, 1.0],
    'reg_lambda': [0.0, 0.1, 1.0]
}

rs = RandomizedSearchCV(
    estimator=lb,
    param_distributions=param_dist,
    n_iter=30,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

rs.fit(X_train, y_train)
print("Best params:", rs.best_params_)
print("CV score:", rs.best_score_)
print("Val accuracy:", accuracy_score(y_val, rs.best_estimator_.predict(X_val)))


Fitting 5 folds for each of 30 candidates, totalling 150 fits
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 2924, number of negative: 2888
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000689 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1372
[LightGBM] [Info] Number of data points in the train set: 5812, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503097 -> initscore=0.012388
[LightGBM] [Info] Start training from score 0.012388
Best params: {'subsample': 1.0, 'reg_lambda': 0.1, 'reg_alpha': 0.0, 'num_leaves': 31, 'n_estimators': 100, 'min_child_samples': 20, 'max_depth': -1, 'learning_rate': 0.05, 'colsample_bytree': 1.0}
CV score: 0.8048843944750874
Val accuracy: 0.7783895388850653


In [75]:
probs = rs.best_estimator_.predict_proba(X_val)[:, 1]
best_threshold = 0.5
best_accuracy = 0.0
for i in range(1, 100):
    threshold = i / 100
    predictions = probs >= threshold
    acc = accuracy_score(y_val, predictions)
    if acc > best_accuracy:
        best_accuracy = acc
        best_threshold = threshold
    # print(f"Threshold: {threshold:.1f}, Accuracy: {acc:.4f}")
print(f"Best threshold: {best_threshold:.2f}, Best accuracy: {best_accuracy:.4f}")

Best threshold: 0.42, Best accuracy: 0.7818


In [83]:
test_probs = rs.best_estimator_.predict_proba(df_test.drop(columns=['PassengerId']))[:, 1]
predictions = test_probs >= best_threshold

print(f"Predictions for test set: {predictions}")
prediction_data=pd.DataFrame({
    'PassengerId': df_test['PassengerId'],
    'Transported': predictions
})

Predictions for test set: [ True False  True ...  True  True  True]


In [85]:
prediction_data.to_csv('./cdata/predictions.csv', index=False)

In [86]:
import joblib

final_model = {
    "model": rs.best_estimator_,
    "threshold": best_threshold
}

joblib.dump(final_model, "./models/final_model.pkl")

['./models/final_model.pkl']

In [87]:
rs.best_estimator_.booster_.save_model("./models/final_model.txt")

In [113]:
import copy
import numpy as np
import onnxmltools
from onnxmltools.convert.common.data_types import FloatTensorType

# Copy the already-trained model
model_for_onnx = copy.deepcopy(rs.best_estimator_)

# ONNX converter cannot handle numpy.bool classes.
# Change only the class metadata in the copy.
model_for_onnx._classes = np.array([0, 1])

# Convert to ONNX
onnx_model = onnxmltools.convert_lightgbm(
    model_for_onnx,
    initial_types=[
        ("float_input", FloatTensorType([None, X_train.shape[1]]))
    ],
    target_opset=15,
    zipmap=False
)

# Save
onnxmltools.utils.save_model(
    onnx_model,
    "./models/final_model.onnx"
)

print("ONNX model saved successfully.")

ONNX model saved successfully.


In [114]:
import onnxruntime as ort

session = ort.InferenceSession("./models/final_model.onnx")

print("Inputs:", session.get_inputs()[0].name)
print("Outputs:", session.get_outputs())

Inputs: float_input
Outputs: [<onnxruntime.capi.onnxruntime_pybind11_state.NodeArg object at 0x000001FEF7BEDC30>, <onnxruntime.capi.onnxruntime_pybind11_state.NodeArg object at 0x000001FEF7BEF6F0>]


In [115]:
for output in session.get_outputs():
    print(
        "Name:", output.name,
        "| Shape:", output.shape,
        "| Type:", output.type
    )

Name: label | Shape: [1] | Type: tensor(int64)
Name: probabilities | Shape: [None, 2] | Type: tensor(float)


In [109]:
df_test.dtypes

PassengerId                      str
Age                          float64
RoomService                  float64
FoodCourt                    float64
ShoppingMall                 float64
Spa                          float64
VRDeck                       float64
HomePlanet_Europa            float64
HomePlanet_Mars              float64
HomePlanet_Unknown           float64
CryoSleep_True               float64
CryoSleep_Unknown            float64
Destination_PSO J318.5-22    float64
Destination_TRAPPIST-1e      float64
Destination_Unknown          float64
VIP_True                     float64
VIP_Unknown                  float64
dtype: object

In [111]:
for i in df_test.columns:
    if i == 'PassengerId':
        continue
    print(f"Column: {i}, Data Type: {df_test[i].dtype}, Unique Values: {df_test[i].nunique()} min: {df_test[i].min()}, max: {df_test[i].max()} mean: {df_test[i].mean()}, median: {df_test[i].median()} std: {df_test[i].std()} ")

Column: Age, Data Type: float64, Unique Values: 79 min: 0.0, max: 79.0 mean: 28.60158989946224, median: 26.0 std: 14.032628598575778 
Column: RoomService, Data Type: float64, Unique Values: 842 min: 0.0, max: 11567.0 mean: 215.06242693476736, median: 0.0 std: 601.9145029266957 
Column: FoodCourt, Data Type: float64, Unique Values: 902 min: 0.0, max: 25273.0 mean: 428.59223754968434, median: 0.0 std: 1510.1559736490672 
Column: ShoppingMall, Data Type: float64, Unique Values: 715 min: 0.0, max: 8292.0 mean: 173.2331073182137, median: 0.0 std: 554.9917760774163 
Column: Spa, Data Type: float64, Unique Values: 833 min: 0.0, max: 19844.0 mean: 295.89595510872107, median: 0.0 std: 1104.8720177293285 
Column: VRDeck, Data Type: float64, Unique Values: 796 min: 0.0, max: 22272.0 mean: 304.8982931961655, median: 0.0 std: 1235.9918106620878 
Column: HomePlanet_Europa, Data Type: float64, Unique Values: 2 min: 0.0, max: 1.0 mean: 0.2342763619359364, median: 0.0 std: 0.4235952090797654 
Column: H

## Final Summary

### Data Preparation
- Checked dataset shape, columns, and missing values.
- Separated features (`X`) and target (`y`).
- Used a stratified train/validation split.

### Models Tried

| Model | Validation Accuracy |
|---|---:|
| Decision Tree | 0.7337 |
| Random Forest | 0.7729 |
| XGBoost | 0.7681 |
| LightGBM | 0.7770 |
| Gradient Boosting | **0.7805** |
| Hard Voting | 0.7729 |
| Soft Voting | 0.7749 |
| Weighted Hard Voting | 0.7777 |
| Weighted Soft Voting | 0.7770 |

### Hyperparameter Tuning

**Gradient Boosting — GridSearchCV**
- Best CV Accuracy: `0.8023`
- Validation Accuracy: `0.7715`

**LightGBM — RandomizedSearchCV**
- Best CV Accuracy: `0.8049`
- Validation Accuracy: `0.7784`

### Threshold Tuning
- Used `predict_proba()` from the tuned LightGBM model.
- Tested thresholds from `0.01` to `0.99`.
- Best threshold: **0.42**
- Best validation accuracy: **0.7818**

### Final
- Final model: **Tuned LightGBM**
- Classification threshold: **0.42**
- Generated predictions for the test set.
- Saved submission to: `./cdata/predictions.csv`